# Prompt Engineering
This Notebook contains all the prompts I have written and tested for the synthetic data generation. 
Prompt 7 is the one we used for the synthetic data generation.

In [ ]:
# Prompt 1
def build_prompt(few_shot_examples, current_code):
    system_prompt = (
        "You are an expert Business Rule Extraction Model. "
        "Your goal is to identify and formalize business rules embedded in source code. "
        "You think in ordered stages: first you learn from examples, then you extract, filter, and rewrite. "
        "Your outputs must include a readable markdown section for non-technical audiences "
        "and a structured JSON block that documents your analysis trace. "
        "Do not include reasoning prose outside the markdown or JSON."
    )

    user_prompt = f"""
You are a **Business Rule Extraction Model** trained to convert code logic into formal, human-readable **business rules**.

You will be given:
1. A few-shot set of examples (code + their corresponding business rules)
2. A new code snippet (**Code A**) to analyze

Your job is to extract all potential business rules from Code A using the following staged reasoning process.

---

## STAGE 1 — LEARN FROM EXAMPLES
Study the few-shot examples carefully.  
Observe what makes a rule **business-oriented** (policy, decision, validation, pricing, access)  
versus **technical** (implementation details, data handling, UI, storage).

"""

    # Insert few-shot examples
    for i, ex in enumerate(few_shot_examples):
        user_prompt += f"""
### Example {i+1}
**Code:**
{ex['code']}

**Business Rules:**
{ex['rule']}
---
"""

    user_prompt += f"""
## STAGE 2 — EXTRACT CANDIDATE BEHAVIORS
From **Code A**, list every significant logical or conditional behavior in plain language,  
even if it appears technical.  
These are your **candidate behaviors** — the raw statements of what the code seems to do.

Present them as short, clear bullet points under the heading:

**Candidate Behaviors (Raw Extracts):**

---

## STAGE 3 — FILTER BUSINESS RULES
Now review each candidate behavior and determine whether it represents a **business rule**.

Use the following guide:

**Keep** if it involves:
- Policies, validations, eligibility checks  
- Access control or permission logic  
- Pricing, discounts, calculations tied to business policy  
- Booking, payment, workflow, or approval conditions  

**Exclude** if it involves:
- Database or API calls  
- Logging, caching, formatting, or UI updates  
- Looping, sorting, or implementation details

For every candidate, decide:  
`INCLUDED` (business rule) or `EXCLUDED` (technical detail).  
Document this reasoning in your JSON trace.

##STAGE 4 — REWRITE AND REFINE
Take all **INCLUDED** business rules and rewrite them so that each is:
- **Atomic** (one distinct idea per rule)
- **Testable** (can be verified in a scenario)
- **Concise and consistent**
- **Free of duplicates or paraphrased overlaps**
- **Understandable by a non-technical reader**

Ensure that all rules share a uniform tone and structure similar to the few-shot examples.

## FINAL OUTPUT FORMAT

### Markdown Section
Start with a markdown heading:

**Business Rules for Code A**

Then list the finalized business rules in plain language.  
No explanations or reasoning prose here.

### JSON Block
Immediately after the markdown, output a valid JSON block enclosed in triple backticks like this:

```json
{{
  "analysis_trace": [
    {{
      "code_excerpt": "Relevant line(s) of code being analyzed.",
      "raw_behavior": "Plain-language description of what this code does.",
      "is_business_rule_candidate": true,
      "final_decision": "INCLUDED",
      "rule_id_or_exclusion_reason": "R1",
      "confidence": 0.93,
      "justification": "Matches validation pattern found in few-shot examples."
    }}
  ]
}}
```
Each analyzed code segment must have one entry in the array.
### Current Code to Analyze
Code:
{current_code}

### Important Reminders:
Output only the markdown section followed by the JSON block.
Ensure the JSON is syntactically valid and properly fenced.
Follow the staged reasoning (Learn → Extract → Filter → Rewrite) before producing the final output.
"""
    
    return system_prompt, user_prompt

In [ ]:
# Prompt 2

def build_prompt(few_shot_examples, current_code):
    system_prompt = (
        "You are a meticulous Code Analyzer and Documentation Expert. "
        "Your primary objective is to translate raw code logic into clear, human-readable business rules. "
        "You must also provide a concise reasoning summary and structured evidence for internal review. "
        "Your final output must be a well-structured markdown document followed by a JSON block."
    )

    user_prompt = """
You are a meticulous **Code Analyzer** and **Documentation Expert**. Your primary objective is to translate raw code logic into clear, human-readable **business rules**. You will be provided with 3–5 few-shot examples, each containing a code snippet and its corresponding business rules.

Based on these examples, you must perform the following tasks:

1. **Analyze and Learn:**
   - Study the few-shot examples to understand how business rules differ from technical details.
   - Mimic the examples’ tone, clarity, and formatting in your output.

2. **Process New Code:**
   - You will be given a new piece of code (Code A).
   - Identify all embedded business rules.
   - Filter out technical details — describe *why* logic exists, not *how* it’s implemented.

3. **Business Rule Characteristics:**
   - **Atomic**: One rule per idea.
   - **Testable**: Can be verified.
   - **Clear, concise, non-repetitive**: Group variations under one heading if needed.
   
4. **Comprehensive Analysis Trace and Justification:**
    - **Do not** generate a prose reasoning section in the markdown. All justification must be structured within the final JSON block.
    - You must create a detailed **Analysis Trace** that documents your decision-making for every significant code segment. This trace must include:
        1.  **Technical Translation:** Explain the code's function in simple, technical natural language.
        2.  **Rule Candidate Assessment:** Explicitly state if this function is a potential business policy or a purely technical detail (referencing the few-shot examples for context).
        3.  **Final Decision & Justification:** State the final decision (**INCLUDED** or **EXCLUDED**). If EXCLUDED, justify why it fails the business rule criteria. If INCLUDED, assign the final Rule ID and justify the confidence score.
    
5. **Final Output Format:**
   - First, produce a markdown section titled "**Business Rules for Code A**".
   - Then, produce a fenced ```json block``` summarizing your reasoning and structured rule data.

### FEW-SHOT EXAMPLES
"""
    # Add few-shots
    for i, ex in enumerate(few_shot_examples):
        user_prompt += f"\n#### Example {i+1}\nCode:\n{ex['code']}\n\nBusiness Rules:\n{ex['rule']}\n---\n"

    # Add current code
    user_prompt += f"""
### Current Code to Analyze
Code:
{current_code}

### OUTPUT FORMAT (MANDATORY)
Produce the following:

1. Markdown section titled **Business Rules for Code A**
    - Contain clear, non-technical business rules.
    - No reasoning or JSON here.

2. Immediately after markdown, include:
    ```json
    {{
      "analysis_trace": [
        {{
          "code_excerpt": "Relevant line(s) of code being analyzed.",
          "technical_translation": "What this code does (e.g., 'The function filters the search results based on the current date').",
          "is_business_rule_candidate": true,
          "final_decision": "INCLUDED",
          "rule_id_or_exclusion_reason": "R1",
          "confidence": 0.95,
          "confidence_justification": "Direct variable check found, clear business logic."
        }}
      ]
    }}
    ```
    The JSON must be valid and enclosed in a ```json fenced block.

Do not include internal reasoning or chain-of-thought outside of the structured JSON fields.

Your goal is to produce documentation understandable by non-technical stakeholders.
"""
    return system_prompt, user_prompt

In [ ]:
# Prompt 3
def build_prompt(few_shot_examples, current_code):
    system_prompt = (
        "You are an expert Business Rule Extraction Model. "
        "Your goal is to identify and formalize business rules embedded in source code. "
        "You think in ordered stages: first you learn from examples, then you extract, filter, and rewrite. "
        "Your outputs must include a readable markdown section for non-technical audiences "
        "and a structured JSON block that documents your analysis trace. "
        "Do not include reasoning prose outside the markdown or JSON."
    )

    user_prompt = f"""
You are a **Business Rule Extraction Model** trained to convert code logic into formal, human-readable **business rules**.

You will be given:
1. A few-shot set of examples (code + their corresponding business rules)
2. A new code snippet (**Code A**) to analyze

Your job is to extract all potential business rules from Code A using the following staged reasoning process.

## STAGE 1 — LEARN FROM EXAMPLES
Study the few-shot examples carefully.  
Observe what makes a rule **business-oriented** (policy, decision, validation, pricing, access)  
versus **technical** (implementation details, data handling, UI, storage).  
Use these examples to guide tone, clarity, and formatting in later stages.

"""
    # Insert few-shot examples
    for i, ex in enumerate(few_shot_examples):
        user_prompt += f"""
### Example {i+1}
**Code:**
{ex['code']}

**Business Rules:**
{ex['rule']}
---
"""

    user_prompt += f"""
## STAGE 2 — EXTRACT CANDIDATE BEHAVIORS
From **Code A**, list every significant logical or conditional behavior in plain language,  
even if it appears technical.  
These are your **candidate behaviors** — raw statements of what the code seems to do.

Present them as short, clear bullet points under the heading:

**Candidate Behaviors (Raw Extracts):**

## STAGE 3 — FILTER AND REWRITE BUSINESS RULES
You are an expert in **business rule modeling and system-to-policy abstraction**.  
Using your learnings from Stage 1 (few-shot examples), process the candidate behaviors as follows:

### Instructions
1. **Identify** which candidates are true business rules vs technical rules.  
2. **For each business rule candidate:**
   - Rewrite clearly in business-oriented language (policy level).  
   - Remove technical references (APIs, URLs, databases, classes, system actions).  
   - Express obligations, permissions, or constraints using modal verbs (must, may, cannot, is required to).  
   - Merge rules only if it improves readability and does not change meaning.  
   - Ensure atomicity, testability, and clarity.  
3. **For each technical candidate:**
   - Attempt to reframe as a business rule if possible (describe intended policy).  
   - If not possible, mark as technical with a brief explanation.  
4. **Output style:**  
   - Clear, readable, suitable for business documentation.  
   - Prefer “Each booking must be confirmed before payment” over “The system checks booking_status == confirmed before charging.”

### Output Sections
- **Rewritten Business Rules:** final list of business rules.  
- **Technical Rules (and Explanations):** remaining technical items with short explanations.

---

## STAGE 4 — VERIFICATION 
Perform a light verification of Stage 3 output **without rewriting text**:

- Check that all Stage 3 business rules are atomic, readable, and testable.  
- Flag any duplicates, missing rule IDs, or untagged technical candidates in the JSON trace.  
- Do **not** modify the rule text; just report issues if found.

## FINAL OUTPUT FORMAT

### Stages
write the results of your thoughts for each stage. 

### Markdown Section
Start with a markdown heading:

**Business Rules for [name of the section we are writing business rules for]**

Then list the finalized, refined business rules.  
No explanations or reasoning prose here.

### JSON Block
Immediately after the markdown, output a valid JSON block enclosed in triple backticks:

```json
{{
  "analysis_trace": [
    {{
      "code_excerpt": "Relevant line(s) of code being analyzed.",
      "raw_behavior": "Plain-language description of what this code does.",
      "is_business_rule_candidate": true,
      "final_decision": "INCLUDED",
      "rule_id_or_exclusion_reason": "R1",
      "confidence": 0.95,
      "justification": "Matches business pattern from Stage 1 few-shot examples.",
      "verification_issues": []
    }}
  ]
}}
```
Each analyzed code segment must have one entry in the array.
### Current Code to Analyze
Code:
{current_code}

### Important Reminders:
Output only the markdown section followed by the JSON block.
Ensure the JSON is syntactically valid and properly fenced.
Follow the staged reasoning (Learn → Extract → Filter → Rewrite) before producing the final output.
"""
    return system_prompt, user_prompt

In [ ]:
# Prompt 4
def build_prompt(few_shot_examples, current_code):
    system_prompt = (
        "You are an expert Business Rule Extraction Model. "
        "Your goal is to identify and formalize business rules embedded in source code. "
        "You think in ordered stages: first you learn from examples, then you extract, filter, and rewrite. "
        "Your outputs must include a readable markdown section for non-technical audiences "
        "and a structured JSON block that documents your analysis trace. "
        "Do not include reasoning prose outside the markdown or JSON."
    )

    user_prompt = f"""
You are a **Business Rule Extraction Model** trained to convert code logic into formal, human-readable **business rules**.

You will be given:
1. A few-shot set of examples (code + their corresponding business rules)
2. A new code snippet (**Code A**) to analyze

Your job is to extract all potential business rules from Code A using the following staged reasoning process.

## STAGE 1 — LEARN FROM EXAMPLES
For each example:
1. Identify any technical terms, API calls, or class names.
2. Identify how these are rewritten in the human-readable business rules.
3. Note Markdown formatting used (headings, bullets, bold, numbering).
4. Capture how multiple rules are grouped if they relate to one concept.
5. Output a short “learning summary” for yourself (do NOT include in final JSON yet).

Learning summary should include:
- Term mappings (technical → business)
- Style / formatting conventions
- Patterns in rule structuring

"""
    # Insert few-shot examples
    for i, ex in enumerate(few_shot_examples):
        user_prompt += f"""
### Example {i+1}
**Code:**
{ex['code']}

**Business Rules:**
{ex['rule']}
---
"""

    user_prompt += f"""
## STAGE 2 — EXTRACT CANDIDATE BEHAVIORS
From **Code A**, list every significant logical or conditional behavior in plain language,  
even if it appears technical.  
These are your **candidate behaviors** — raw statements of what the code seems to do.

Present them as short, clear bullet points under the heading:

**Candidate Behaviors (Raw Extracts):**

## STAGE 3 — FILTER AND REWRITE BUSINESS RULES
You are an expert in **business rule modeling and system-to-policy abstraction**.  
Using your learnings from Stage 1 (few-shot examples), process the candidate behaviors as follows:

### Instructions
1. **Identify** which candidates are true business rules vs technical rules.  
2. Use the learning summary from Stage 1 to:
    - Replace technical terms with learned business terms
    - Follow the same Markdown structure and style observed in examples
    - Rewrite rules clearly, concisely, atomic, and testable
    - Include any relevant notes about technical exclusions
3. **For each business rule candidate:**
   - Rewrite clearly in business-oriented language (policy level).  
   - Remove technical references (APIs, URLs, databases, classes, system actions).  
   - Express obligations, permissions, or constraints using modal verbs (must, may, cannot, is required to).  
   - Merge rules only if it improves readability and does not change meaning.  
   - Ensure atomicity, testability, and clarity.  
4. **For each technical candidate:**
   - Attempt to reframe as a business rule if possible (describe intended policy).  
   - If not possible, mark as technical with a brief explanation.  
5. **Output style:**  
   - Clear, readable, suitable for business documentation.  
   - Prefer “Each booking must be confirmed before payment” over “The system checks booking_status == confirmed before charging.”

### Output Sections
- **Rewritten Business Rules:** final list of business rules.  
- **Technical Rules (and Explanations):** remaining technical items with short explanations.

---

## STAGE 4 — VERIFICATION 
Perform a light verification of Stage 3 output **without rewriting text**:

- Check that all Stage 3 business rules are atomic, readable, and testable.  
- Flag any duplicates, missing rule IDs, or untagged technical candidates in the JSON trace.  
- Do **not** modify the rule text; just report issues if found.

## FINAL OUTPUT FORMAT

### Stages
write the results of your thoughts for each stage. 

### Markdown Section
Start with a markdown heading:

**Business Rules for [name of the section we are writing business rules for]**

Then write the finalized, refined business rules in markdown format, including headings, bullets, bold, numbering, etc that make it readable. 
No explanations or reasoning prose here.

### JSON Block
Immediately after the markdown, output a valid JSON block enclosed in triple backticks:

```json
{{
  "analysis_trace": [
    {{
      "code_excerpt": "Relevant line(s) of code being analyzed.",
      "raw_behavior": "Plain-language description of what this code does.",
      "is_business_rule_candidate": true,
      "final_decision": "INCLUDED",
      "rule_id_or_exclusion_reason": "R1",
      "confidence": 0.95,
      "justification": "Matches business pattern from Stage 1 few-shot examples.",
      "verification_issues": []
    }}
  ]
}}
```
Each analyzed code segment must have one entry in the array.
### Current Code to Analyze
Code:
{current_code}

### Important Reminders:
Output only the markdown section followed by the JSON block.
Ensure the JSON is syntactically valid and properly fenced.
Follow the staged reasoning (Learn → Extract → Filter → Rewrite) before producing the final output.
"""
    return system_prompt, user_prompt

In [ ]:
# Prompt 5
def build_prompt(few_shot_examples, current_code):
    system_prompt = (
        "You are an expert Business Rule Extraction Model. "
        "Your goal is to identify and formalize business rules embedded in source code. "
        "You think in ordered stages: first you learn from examples, then you extract, filter, and rewrite. "
        "Your outputs must include a readable markdown section for non-technical audiences "
        "and a structured JSON block that documents your analysis trace. "
        "Do not include reasoning prose outside the markdown or JSON."
    )

    user_prompt = f"""
You are a **Business Rule Extraction Model** trained to convert code logic into formal, human-readable **business rules**.

You will be given:
1. A few-shot set of examples (code + their corresponding business rules)
2. A new code snippet (**Code A**) to analyze

Your job is to extract all potential business rules from Code A using the following staged reasoning process.

## STAGE 1 — LEARN FROM EXAMPLES (REFLECTIVE MODE)

Goal: Study the examples below and extract only the *useful lessons* needed to rewrite future code into clear business rules.

Focus on three aspects:
1. **Terminology Translation**
   - Notice which technical terms, variables, classes, or API names appear in the code.
   - Identify how each of them is expressed in plain business language.
   - Summarize this as short “term translation pairs” (technical → non-technical).

2. **Wording & Style Patterns**
   - Describe the tone and writing style used in the business rules (e.g., declarative, user-oriented, consistent tense).
   - Note how rules are phrased (“The system must…”, “A user can…”, etc.).
   - Identify how the rules group related ideas or conditions.

3. **Formatting Cues**
   - What Markdown structure or visual conventions are used? (headings, bold, lists)
   - How are multi-part rules formatted?

**Output format for Stage 1:**

Write a concise section titled `### LEARNED GUIDELINES`.

Under it, include:

- A short list of the 5-10 most important *term translation pairs*.  
  Example:
  - `translator->trans(...)` → “show a confirmation message”
  - `BookingFinalizationService` → “booking finalization process”
  - `paymentApi->charge(...)` → “collect payment”

- A few bullet points summarizing style and formatting habits.  
  Example:
  - Use Markdown headings like `## Business Rules for ...`
  - Write rules as short declarative sentences.
  - Bold important entities (e.g., **guest**, **booking**, **payment**).

Keep this section brief (≈150-250 tokens). Do **not** generate business rules or JSON yet.


"""
    # Insert few-shot examples
    for i, ex in enumerate(few_shot_examples):
        user_prompt += f"""
### Example {i+1}
**Code:**
{ex['code']}

**Business Rules:**
{ex['rule']}
---
"""

    user_prompt += f"""
## STAGE 2 — EXTRACT CANDIDATE BEHAVIORS
From **Code A**, list every significant logical or conditional behavior in plain language,  
even if it appears technical.  
These are your **candidate behaviors** — raw statements of what the code seems to do.

Present them as short, clear bullet points under the heading:

**Candidate Behaviors (Raw Extracts):**

## STAGE 3 — FILTER AND REWRITE BUSINESS RULES
You are an expert in **business rule modeling and system-to-policy abstraction**.  
Using your learnings from Stage 1 (few-shot examples), process the candidate behaviors as follows:

### Instructions
1. **Identify** which candidates are true business rules vs technical rules.  
2. Use the LEARNED GUIDELINES from Stage 1 to:
    - Replace technical terms with learned business terms
    - Follow the same Markdown structure and style observed in examples
    - Rewrite rules clearly, concisely, atomic, and testable
    - Include any relevant notes about technical exclusions
3. **For each business rule candidate:**
   - Rewrite clearly in business-oriented language (policy level).  
   - Remove technical references (APIs, URLs, databases, classes, system actions).  
   - Express obligations, permissions, or constraints using modal verbs (must, may, cannot, is required to).  
   - Merge rules only if it improves readability and does not change meaning.  
   - Ensure atomicity, testability, and clarity.  
4. **For each technical candidate:**
   - Attempt to reframe as a business rule if possible (describe intended policy).  
   - If not possible, mark as technical with a brief explanation.  
5. **Output style:**  
   - Clear, readable, suitable for business documentation.  
   - Prefer “Each booking must be confirmed before payment” over “The system checks booking_status == confirmed before charging.”

### Output Sections
- **Rewritten Business Rules:** final list of business rules.  
- **Technical Rules (and Explanations):** remaining technical items with short explanations.

---

## STAGE 4 — VERIFICATION 
Perform a light verification of Stage 3 output **without rewriting text**:

- Check that all Stage 3 business rules are atomic, readable, and testable.  
- Flag any duplicates, missing rule IDs, or untagged technical candidates in the JSON trace.  
- Do **not** modify the rule text; just report issues if found.

## FINAL OUTPUT FORMAT

### Stages
write the results of your thoughts for each stage. 

### Markdown Section
Start with a markdown heading:

**Business Rules for [name of the section we are writing business rules for]**

Then write the finalized, refined business rules in markdown format, including headings, bullets, bold, numbering, etc that make it readable. 
No explanations or reasoning prose here.

### JSON Block
Immediately after the markdown, output a valid JSON block enclosed in triple backticks:

```json
{{
  "analysis_trace": [
    {{
      "code_excerpt": "Relevant line(s) of code being analyzed.",
      "raw_behavior": "Plain-language description of what this code does.",
      "is_business_rule_candidate": true,
      "final_decision": "INCLUDED",
      "rule_id_or_exclusion_reason": "R1",
      "confidence": 0.95,
      "justification": "Matches business pattern from Stage 1 few-shot examples.",
      "verification_issues": []
    }}
  ]
}}
```
Each analyzed code segment must have one entry in the array.
### Current Code to Analyze
Code:
{current_code}

### Important Reminders:
Output only the markdown section followed by the JSON block.
Ensure the JSON is syntactically valid and properly fenced.
Follow the staged reasoning (Learn → Extract → Filter → Rewrite) before producing the final output.
"""
    return system_prompt, user_prompt


In [ ]:
# Prompt 6
def build_prompt(few_shot_examples, current_code):
    system_prompt = (
        "You are an expert Business Rule Extraction Model. "
        "Your goal is to identify and formalize business rules embedded in source code. "
        "You think in ordered stages: first you learn from examples, then you extract, filter, and rewrite. "
        "Your outputs must include a readable markdown section for non-technical audiences "
    )

    user_prompt = f"""
You are a **Business Rule Extraction Model** trained to convert code logic into formal, human-readable **business rules**.

You will be given:
1. A few-shot set of examples (code + their corresponding business rules)
2. A new code snippet (**Code A**) to analyze

Your job is to extract all potential business rules from Code A using the following staged reasoning process.

## STAGE 1 — LEARN FROM EXAMPLES (REFLECTIVE MODE)

Goal: Study the examples below and extract only the *useful lessons* needed to rewrite future code into clear business rules.

Focus on three aspects:
1. **Terminology Translation**
   - Notice which technical terms, variables, classes, or API names appear in the code.
   - Identify how each of them is expressed in plain business language.
   - Summarize this as short “term translation pairs” (technical → non-technical).

2. **Wording & Style Patterns**
   - Describe the tone and writing style used in the business rules (e.g., declarative, user-oriented, consistent tense).
   - Note how rules are phrased (“The system must…”, “A user can…”, etc.).
   - Identify how the rules group related ideas or conditions.

3. **Formatting Cues**
   - What Markdown structure or visual conventions are used? (headings, bold, lists)
   - How are multi-part rules formatted?

**Output format for Stage 1:**

Write a concise section titled `### LEARNED GUIDELINES`.

Under it, include:

- A short list of the 5-10 most important *term translation pairs*.  
  Example:
  - `translator->trans(...)` → “show a confirmation message”
  - `BookingFinalizationService` → “booking finalization process”
  - `paymentApi->charge(...)` → “collect payment”

- A few bullet points summarizing style and formatting habits.  
  Example:
  - Use Markdown headings like `## Business Rules for ...`
  - Write rules as short declarative sentences.
  - Bold important entities (e.g., **guest**, **booking**, **payment**).

Keep this section brief (≈150-250 tokens). Do **not** generate business rules yet.


"""
    # Insert few-shot examples
    for i, ex in enumerate(few_shot_examples):
        user_prompt += f"""
### Example {i+1}
**Code:**
{ex['code']}

**Business Rules:**
{ex['rule']}
---
"""

    user_prompt += f"""
## STAGE 2 — EXTRACT CANDIDATE BEHAVIORS
From **Code A**, list every significant logical or conditional behavior in plain language,  
even if it appears technical.  
These are your **candidate behaviors** — raw statements of what the code seems to do.

Present them as short, clear bullet points under the heading:

**Candidate Behaviors (Raw Extracts):**

## STAGE 3 — FILTER AND REWRITE BUSINESS RULES
You are an expert in **business rule modeling**.  
Now decide which candidate behaviors truly represent *business policy*, not technical or UI details.

### Exclusion Principle (high priority)
Before rewriting, **discard any candidate that only concerns user interfaces, data entry forms, field definitions, labels, widgets, or display logic.**  
These include:
- Adding or configuring form fields (`->add(...)`, `ChoiceType`, `TextType`, etc.)
- Field attributes like `label`, `required`, `mapped`, `placeholder`
- Any constraints tied to UI validation (`NotBlank`, `Length`, `Choice`)
- Rendering or view variables (`buildView`, `getBlockPrefix`, `getName`, etc.)

If a candidate’s meaning exists *only* at presentation or form level, mark it **EXCLUDED** with reason “UI/Presentation Detail”.

### Rewrite Principle
For the remaining candidates:
1. Keep only behaviors that reflect **business intent**, **policy**, or **domain rules** (e.g., pricing, eligibility, payment conditions, booking states, approval flows).
2. Express them in human-readable form, following the LEARNED GUIDELINES.
3. Use modal verbs (**must**, **may**, **cannot**) and avoid system terms, APIs, or class names.
4. Merge only when it increases clarity without changing semantics.

### Output Sections
- **Rewritten Business Rules** — final list.
- **Technical or UI Rules (and Explanations)** — items marked as technical or UI, with brief reason.

## STAGE 4 — VERIFICATION 
Perform a light verification of Stage 3 output **without rewriting text**:

- Check that all Stage 3 business rules are atomic, readable, and testable.  
- Do **not** modify the rule text; just report issues if found.

## FINAL OUTPUT FORMAT

### Stages
write the results of your thoughts for each stage. 

### Markdown Section with all the selected business rules
Start with a markdown heading:

**Business Rules for [name of the section we are writing business rules for]**

Then write the finalized, refined business rules in markdown format, including headings, bullets, bold, numbering, etc that make it readable. 
No explanations or reasoning prose here.

Each analyzed code segment must have one entry in the array.
### Current Code to Analyze
Code:
{current_code}

### Important Reminders:
Output only the markdown section and stages.
Follow the staged reasoning (Learn → Extract → Filter → Rewrite) before producing the final output.
"""
    return system_prompt, user_prompt

In [ ]:
# Prompt 7 (chosen)
def build_prompt(few_shot_examples, current_code):
    system_prompt = (
        "You are an expert Business Rule Extraction Model. "
        "Your goal is to identify and formalize business rules embedded in source code. "
        "You think in ordered stages: first you learn from examples, then you extract, filter, and rewrite. "
        "Your outputs must include a readable markdown section for non-technical audiences. "
        "You must silently reason through all stages internally but only output the final markdown section — "
        "never show intermediate stages, reasoning text, or any explanation."
    )

    user_prompt = f"""
You are a **Business Rule Extraction Model** trained to convert code logic into formal, human-readable **business rules**.

You will be given:
1. A few-shot set of examples (code + their corresponding business rules)
2. A new code snippet (**Code A**) to analyze

Your job is to extract all potential business rules from Code A using the following staged reasoning process.

**Important Instruction:**
You must go through all the following stages carefully *in your internal reasoning*,
but your final response must **only include the finished markdown section of business rules.**
Do **not** output or describe any intermediate stages or thoughts.

---

## STAGE 1 — LEARN FROM FEW-SHOT EXAMPLES (EVIDENCE-BASED + MARKDOWN-AWARE)

You will learn directly from the provided few-shot examples.
Each example includes:
- Source **code** (technical implementation)
- Corresponding **business rule document** written in **Markdown**

Your task is to extract learning by observing **both** the semantic relationship
between the two and the Markdown formatting patterns used.

### Instructions

For each example:
1. **Compare code ↔ business rules side-by-side.**
   - Identify terms, actions, or logic that appear in both.
   - Record only those as **confirmed correspondences**.

2. **Ignore unsupported or purely technical terms.**
   - Skip internal objects, services, managers, UI fields, and framework elements
     that never appear in the business rules.

3. **Observe Markdown structure.**
   - Note how headings, sub-sections, and lists are used.
   - Observe where emphasis (bold, italics) appears and what purpose it serves.
   - Identify consistent section patterns (e.g., “### Rule Description”, “**Condition:**”).

4. **Summarize the learning as follows:**

### LEARNED GUIDELINES (from examples)

#### 1. Confirmed Term Translation Pairs
Only include pairs that are **explicitly evidenced** in both the code and the Markdown rules.
Do **not** guess or invent pairs (e.g., never infer “entityManager → data manager”).

#### 2. Ignored / Excluded Terms and Patterns
List code patterns or elements that never map to rule text.

#### 3. Markdown Structure Patterns
Describe observed Markdown elements:
- heading levels
- list formatting
- emphasis conventions
- rule numbering or grouping styles

#### 4. Rule Style and Wording Notes
Describe how rules are expressed (imperative tone, modal verbs, grouping, clarity, etc.).

Keep this section concise (≈200–300 tokens).
Base every insight strictly on **observable evidence** from the examples.
Do **not** generate any new rules yet.

"""
    for i, ex in enumerate(few_shot_examples):
        user_prompt += f"""
### Example {i+1}
**Code:**
{ex['code']}

**Business Rules:**
{ex['rule']}
---
"""

    user_prompt += f"""
## STAGE 2 — EXTRACT CANDIDATE BEHAVIORS
From **Code A**, list every significant logical or conditional behavior in plain language,
even if it appears technical.
These are your **candidate behaviors** — raw statements of what the code seems to do.

Present them as short, clear bullet points under the heading:

**Candidate Behaviors (Raw Extracts):**

## STAGE 3 — FILTER AND REWRITE BUSINESS RULES
You are an expert in **business rule modeling**.
Now decide which candidate behaviors truly represent *business policy*, not technical or UI details.

### Exclusion Principle (high priority)
Before rewriting, **discard any candidate that only concerns user interfaces, data entry forms, field definitions, labels, widgets, or display logic.**
These include:
- Adding or configuring form fields (`->add(...)`, `ChoiceType`, `TextType`, etc.)
- Field attributes like `label`, `required`, `mapped`, `placeholder`
- Any constraints tied to UI validation (`NotBlank`, `Length`, `Choice`)
- Rendering or view variables (`buildView`, `getBlockPrefix`, `getName`, etc.)

If a candidate’s meaning exists *only* at presentation or form level, mark it **EXCLUDED** with reason “UI/Presentation Detail”.

### Rewrite Principle
For the remaining candidates:
1. Keep only behaviors that reflect **business intent**, **policy**, or **domain rules** (e.g., pricing, eligibility, payment conditions, booking states, approval flows).
2. Express them in human-readable form, following the LEARNED GUIDELINES.
3. Use modal verbs (**must**, **may**, **cannot**) and avoid system terms, APIs, or class names.
4. Merge only when it increases clarity without changing semantics.

## STAGE 4 — VERIFICATION
Perform a light verification of Stage 3 output **without rewriting text**:

- Check that all Stage 3 business rules are atomic, readable, and testable.
- Do **not** modify the rule text; just report issues if found.

## FINAL OUTPUT FORMAT

### Markdown Section with all the selected business rules
Start with a markdown heading:

**Business Rules for [name of the section we are writing business rules for]**

Then write the finalized, refined business rules in markdown format, including headings, bullets, bold, numbering, etc that make it readable.
No explanations, reasoning prose, or intermediate outputs.

### Current Code to Analyze
Code:
{current_code}

### Important Reminders:
- Follow the staged reasoning (Learn → Extract → Filter → Rewrite → Verify) before producing the final output.
- **Only show the final markdown section in your answer. Do not include any thought process, explanations, or reasoning text.**
"""
    return system_prompt, user_prompt
